# APUNTES PRÁCTICOS - EXAMEN DE ML

## **Preparación de Datos**
0. 
   - Importaciones Generales
   - pandas, numpy, matplotlib, seaborn + sklearn modules
   
1. 
   - Carga y Exploración de Datos
   - pd.read_csv() | df.head() | df.info() | df.isna().sum()
   - Limpieza estructural (df.drop())

2. 
   - Imputación y Codificación
   - SimpleImputer (mean, most_frequent)
   - LabelEncoder + pd.get_dummies()

## **Análisis Exploratorio**
3. 
   - Matriz de Correlación
   - df.corr() | sns.heatmap() | Ordenación por Target

## **Preparación del Modelo**
4. 
   - División y Escalado
   - train_test_split() | StandardScaler (fit_transform / transform)

## **Modelos Supervisados**
5. 
   - Clasificación
   - RandomForestClassifier | SVC | MLPClassifier | KNeighborsClassifier | DecisionTreeClassifier | LogisticRegression
   - classification_report | confusion_matrix | accuracy_score

6. 
   - Regresión
   - LinearRegression
   - MAE | MSE | R² Score

## **Modelos No Supervisados**
7. 
   - Agrupamiento (K-Means y Jerárquico)
   - KMeans.fit_predict() | AgglomerativeClustering | silhouette_score

8. 
   - Reducción de Dimensionalidad (PCA)
   - PCA(n_components=2) | Varianza conservada | Visualización 2D

## Libreias principales

**Sirve para:** Cargar todas las herramientas matemáticas y visuales antes de empezar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Librerías principales cargadas correctamente.")

## BLOQUE 1: CARGA Y LIMPIEZA ESTRUCTURAL

**Sirve para:** Cargar los datos y eliminar identificadores que confunden al modelo.

In [ ]:
# 1. Cargar el archivo
df = pd.read_csv('dataset_examen.csv')

# 2. Eliminar columnas que no aportan información (IDs, nombres, índices)
# Si el modelo intenta aprender del ID del pasajero, colapsará.
df = df.drop(["Unnamed: 0", "id"], axis=1)

# 3. Explorar el dataset
df.head()  # Ver las primeras 5 filas

# Ver información del dataset (tipos, memoria, nulos)
df.info()

# Contar nulos por columna
print("Nulos por columna:")
print(df.isna().sum())

## BLOQUE 2: IMPUTACIÓN Y CODIFICACIÓN (PREPROCESAMIENTO)

**Sirve para:** Limpiar el dataset de nulos y convertir textos a números.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# 1. Rellenar Nulos (Imputación)
# Para números (ej. Edad): Usamos la media
imputador_num = SimpleImputer(strategy='mean')
df[['ColumnaNum1', 'ColumnaNum2']] = imputador_num.fit_transform(df[['ColumnaNum1', 'ColumnaNum2']])

# Para texto (ej. Ciudad): Usamos la moda (el más frecuente)
imputador_cat = SimpleImputer(strategy='most_frequent')
df[['ColumnaTexto']] = imputador_cat.fit_transform(df[['ColumnaTexto']])

# 2. Codificación (Texto a Número)
# Variable objetivo (Target): Usamos LabelEncoder
le = LabelEncoder()
df['satisfaction'] = le.fit_transform(df['satisfaction'])

# Variables de entrada categóricas sin orden (Nominales): Usamos One-Hot Encoding
df = pd.get_dummies(df, columns=['ColumnaTexto1', 'ColumnaTexto2'])

## BLOQUE 3: MATRIZ DE CORRELACIÓN

**Sirve para:** Identificar matemáticamente qué variable influye más en el objetivo.

⚠️ **IMPORTANTE:** Solo se puede hacer cuando TODO el dataset ya son números.

In [ ]:
plt.figure(figsize=(12, 8))
matriz_corr = df.corr()
sns.heatmap(matriz_corr, annot=False, cmap='coolwarm')
plt.title('Matriz de Correlación de Variables')
plt.show()

# Ver exactamente qué variable está más pegada al 'Target' (satisfaction):
print("Correlación directa con la variable objetivo:")
print(matriz_corr['satisfaction'].sort_values(ascending=False))

## BLOQUE 4: DIVISIÓN TRAIN/TEST Y ESCALADO

**Sirve para:** Separar los datos y dejarlos en la misma escala (media 0, varianza 1).

In [ ]:
X = df.drop('satisfaction', axis=1) # Todas las columnas menos el Target
y = df['satisfaction']              # Solo el Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Aprende y transforma Train
X_test_scaled = scaler.transform(X_test)       # SOLO transforma Test

## BLOQUE 5: APRENDIZAJE SUPERVISADO - CLASIFICACIÓN

**Sirve para:** Predecir categorías y evaluar con la Matriz de Confusión.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Descomentar el modelo que pida el examen:
# RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
modelo = RandomForestClassifier(random_state=42)
# SVC(kernel='rbf', C=1.0, random_state=42) | kernel: 'linear', 'poly', 'rbf', 'sigmoid'
modeloSVC = SVC(kernel='rbf', random_state=42) 
# MLPClassifier(hidden_layer_sizes=(100, 50), activation='relu', alpha=0.0001, learning_rate_init=0.001, max_iter=500, random_state=42)
modeloMLPC = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42) 
# KNeighborsClassifier(n_neighbors=5, weights='distance')
modeloKNN = KNeighborsClassifier(n_neighbors=5)
# DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2, max_features='sqrt', random_state=42)
modeloDecTree = DecisionTreeClassifier(random_state=42)
# LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', max_iter=100, random_state=42)
modeloLReg = LogisticRegression(random_state=42)

# Entrenamiento y Predicción (Seleccionar el modelo necesario)
modelo.fit(X_train_scaled, y_train)
predicciones = modelo.predict(X_test_scaled)

# Evaluación
print(classification_report(y_test, predicciones))
sns.heatmap(confusion_matrix(y_test, predicciones), annot=True, cmap='Blues', fmt='g')
plt.title('Matriz de Confusión')
plt.show()

In [ ]:
#Visualizacion Arbol
from sklearn.tree import plot_tree
plt.figure(figsize=(15, 10))
plot_tree(modelo, filled=True, feature_names=X.columns, class_names=True, max_depth=3)
plt.title("Estructura Lógica del Árbol de Decisión")
plt.show()

## BLOQUE 6: APRENDIZAJE SUPERVISADO - REGRESIÓN

**Sirve para:** Predecir números continuos y evaluar sus errores.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

modelo_reg = LinearRegression()
modelo_reg.fit(X_train_scaled, y_train)
predicciones_reg = modelo_reg.predict(X_test_scaled)

mae = mean_absolute_error(y_test, predicciones_reg)
mse = mean_squared_error(y_test, predicciones_reg)
r2 = r2_score(y_test, predicciones_reg) 
print(f"MAE: {mae:.2f} | MSE: {mse:.2f} | R2 Score: {r2:.2f}")

## BLOQUE 7: APRENDIZAJE NO SUPERVISADO - AGRUPAMIENTO (K-MEANS)

**Sirve para:** Encontrar perfiles ocultos cuando no hay variable 'Target'.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# K-Means(n_clusters=3, init='k-means++', n_init=10, max_iter=300, tol=1e-4, random_state=42)
# K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
df['Cluster_KMeans'] = kmeans.fit_predict(X_train_scaled) 

score_silueta_kmeans = silhouette_score(X_train_scaled, df['Cluster_KMeans'])
print(f"K-Means - Coeficiente de Silueta: {score_silueta_kmeans:.2f} (Cercano a 1 es mejor)")
print(f"K-Means - Inercia: {kmeans.inertia_:.2f}")
# AgglomerativeClustering(n_clusters=3, linkage='ward') | linkage: 'ward', 'complete', 'average', 'single'
# Agrupamiento Jerárquico
agglomerative = AgglomerativeClustering(n_clusters=3, linkage='ward')
df['Cluster_Jerarchico'] = agglomerative.fit_predict(X_train_scaled)

score_silueta_jerarquico = silhouette_score(X_train_scaled, df['Cluster_Jerarchico'])
print(f"\nAgrupamiento Jerárquico - Coeficiente de Silueta: {score_silueta_jerarquico:.2f}")

## BLOQUE 8: REDUCCIÓN DE DIMENSIONALIDAD (PCA)

**Sirve para:** Aplastar miles de variables en unas pocas para graficar.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2) 
X_pca = pca.fit_transform(X_train_scaled)

varianza_conservada = np.sum(pca.explained_variance_ratio_)
print(f"Información conservada al reducir a 2D: {varianza_conservada * 100:.2f}%")

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='viridis')
plt.title('Proyección PCA 2D')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.show()